# Sampling Methods

In [5]:
import numpy as np

## Uniform vs. Stratified Sampling 

**Uniform sampling** picks points with equal probability across the entire space. While simple, it often leaves "gaps" or clusters data points awkwardly.

**Stratified sampling** ensures that sub-groups (strata) are represented. In a 1D space $[0, 1]$, this means dividing the range into $N$ equal bins and picking one point per bin. This reduces variance and ensures better coverage of the space.

In [6]:
def sample_1d(n_samples, method="uniform"):
    if method == "uniform":
        # Every point has an equal 1/N chance
        return np.random.uniform(0, 1, n_samples)
    
    elif method == "stratified":
        # Divide [0, 1] into n_samples bins
        bins = np.linspace(0, 1, n_samples + 1)
        low = bins[:-1]
        high = bins[1:]
        # Sample one point within each bin
        return np.random.uniform(low, high)



## Reservoir Sampling 

This is a classic "streaming" algorithm. Use this when you have a dataset so large it doesn't fit in memory, or an incoming stream of unknown size, and you need to maintain a representative sample of size $k$.

The Logic:
1. Fill the "reservoir" with the first $k$ elements.
2. For every subsequent element $i$ (where $i > k$), pick a random integer $j$ between $0$ and $i$.
3. If $j < k$, replace `reservoir[j]` with the new element.

In [7]:
def reservoir_sampling(stream, k):
    # Initialize reservoir with first k elements
    reservoir = np.copy(stream[:k])
    
    for i in range(k, len(stream)):
        # Generate a random index between 0 and i (inclusive)
        j = np.random.randint(0, i + 1)
        
        # If the index falls within the reservoir range, replace it
        if j < k:
            reservoir[j] = stream[i]
            
    return reservoir


## Generating Data from Distributions

In an interview, you might be asked to transform a $Uniform(0, 1)$ distribution into something else.

### Inverse Transform Sampling

If you have a probability density function (PDF) $f(x)$ and its cumulative distribution function (CDF) $F(x)$, you can generate samples by:
- Generating $u \sim Uniform(0, 1)$.
- Computing $x = F^{-1}(u)$.
 
For an Exponential Distribution with rate $\lambda$:

The CDF is $F(x) = 1 - e^{-\lambda x}$.
Solving for $x$ gives the inverse:

$$x = -\frac{\ln(1-u)}{\lambda}$$


In [8]:

def sample_exponential(n_samples, lambd=1.0):
    u = np.random.uniform(0, 1, n_samples)
    # Since 1-u is also Uniform(0,1), we can just use u
    return -np.log(1 - u) / lambd


The **Box-Muller Transform** (Generating Normals)

To generate $Normal(0, 1)$ samples from $Uniform(0, 1)$ without using `np.random.normal`:

$$Z_0 = \sqrt{-2\ln U_1} \cos(2\pi U_2)$$

$$Z_1 = \sqrt{-2\ln U_1} \sin(2\pi U_2)$$

In [9]:

def sample_normal_box_muller(n_samples):
    # n_samples should be even for this implementation
    u1 = np.random.uniform(0, 1, n_samples // 2)
    u2 = np.random.uniform(0, 1, n_samples // 2)
    
    r = np.sqrt(-2 * np.log(u1))
    theta = 2 * np.pi * u2
    
    z0 = r * np.cos(theta)
    z1 = r * np.sin(theta)
    
    return np.concatenate([z0, z1])


With Imbalanced Data (e.g., millions of miles of straight driving vs. a few seconds of a pedestrian jumping into the road):
- Stratified Sampling is used to ensure those rare "edge cases" are included in training batches.
- Reservoir Sampling is used to sample from huge logs of LIDAR data streaming off a vehicle.

# LLM Decoder Sampling Strategies

At each time step, a language model produces **logits** $z_i$ for every token $i$ in the vocabulary. These are converted to probabilities via softmax:

$$P(x_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

The **sampling strategy** decides how to pick the next token from this distribution. The choice profoundly affects output quality, diversity, and whether the model can express **multi-modal** (multiple plausible continuations) outputs.

We'll use a toy vocabulary and a simulated `next_token_logits` function throughout.

In [1]:
import numpy as np
from typing import List, Tuple

# --- Toy vocabulary ---
VOCAB = ["the", "a", "cat", "dog", "sat", "on", "mat", "ran", "big", "small", "<EOS>"]
VOCAB_SIZE = len(VOCAB)
ID2TOK = {i: w for i, w in enumerate(VOCAB)}
TOK2ID = {w: i for i, w in enumerate(VOCAB)}

def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically-stable softmax."""
    e = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def dummy_logits(context: List[int], rng: np.random.Generator = None) -> np.ndarray:
    """Return fake logits for the next token given a context.
    
    Uses a fixed seed derived from context so results are reproducible 
    but still vary with different contexts.
    """
    if rng is None:
        seed = sum(c * (i + 1) for i, c in enumerate(context)) % (2**31)
        rng = np.random.default_rng(seed)
    return rng.standard_normal(VOCAB_SIZE)

np.set_printoptions(precision=4, suppress=True)
print("Vocab:", VOCAB)

Vocab: ['the', 'a', 'cat', 'dog', 'sat', 'on', 'mat', 'ran', 'big', 'small', '<EOS>']


## 1. Greedy Search

At each step, pick the token with the **highest probability**:

$$x_t = \arg\max_i P(x_i \mid x_{<t})$$

**Pros:** Deterministic, fast, simple.  
**Cons:** Always produces the **same** output — completely fails to capture multi-modal distributions. If the true distribution has two equally good continuations ("The cat sat on the **mat**" vs. "The cat sat on the **floor**"), greedy search will always pick one and never explore the other.

In [2]:
def greedy_search(prompt_ids: List[int], max_len: int = 6) -> List[int]:
    """Always pick argmax of the probability distribution."""
    generated = list(prompt_ids)
    eos_id = TOK2ID["<EOS>"]
    
    for _ in range(max_len):
        logits = dummy_logits(generated)
        probs = softmax(logits)
        next_id = int(np.argmax(probs))
        generated.append(next_id)
        if next_id == eos_id:
            break
    return generated

# Run greedy 3 times — always the same output (deterministic)
prompt = [TOK2ID["the"], TOK2ID["big"]]
for run in range(3):
    ids = greedy_search(prompt)
    print(f"Run {run+1}: {[ID2TOK[i] for i in ids]}")

Run 1: ['the', 'big', 'sat', '<EOS>']
Run 2: ['the', 'big', 'sat', '<EOS>']
Run 3: ['the', 'big', 'sat', '<EOS>']


## 2. Beam Search

Maintain the top-$B$ (beam width) partial sequences at each step. For each beam, expand all vocabulary tokens, score them by cumulative **log-probability**, keep the top-$B$ candidates, and repeat.

$$\text{score}(y_{1:t}) = \sum_{i=1}^{t} \log P(y_i \mid y_{<i})$$

**Pros:** Explores a wider search space than greedy; often finds higher-probability complete sequences.  
**Cons:** Still **deterministic** — always returns the same top-B results. Tends to produce generic, high-frequency outputs. Poor at capturing multi-modal diversity because all beams usually converge to similar sequences. Computationally $O(B \times V)$ per step.

In [3]:
def beam_search(prompt_ids: List[int], beam_width: int = 3, max_len: int = 6) -> List[Tuple[float, List[int]]]:
    """
    Return top-B sequences sorted by cumulative log-probability.
    
    Each beam is (log_prob, token_ids).
    """
    eos_id = TOK2ID["<EOS>"]
    # Initialize beams: (cumulative_log_prob, sequence)
    beams = [(0.0, list(prompt_ids))]
    completed = []
    
    for _ in range(max_len):
        candidates = []
        for log_prob, seq in beams:
            if seq[-1] == eos_id:
                completed.append((log_prob, seq))
                continue
            
            logits = dummy_logits(seq)
            probs = softmax(logits)
            log_probs = np.log(probs + 1e-12)
            
            # Expand every token in vocabulary
            for token_id in range(VOCAB_SIZE):
                new_log_prob = log_prob + log_probs[token_id]
                candidates.append((new_log_prob, seq + [token_id]))
        
        if not candidates:
            break
        
        # Keep top-B candidates
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]
    
    # Merge completed and remaining beams
    all_results = completed + beams
    all_results.sort(key=lambda x: x[0], reverse=True)
    return all_results[:beam_width]

# Run beam search
results = beam_search(prompt, beam_width=3)
for i, (score, ids) in enumerate(results):
    tokens = [ID2TOK[i] for i in ids]
    print(f"Beam {i+1} (log-prob={score:.3f}): {tokens}")

Beam 1 (log-prob=-2.693): ['the', 'big', 'sat', '<EOS>']
Beam 2 (log-prob=-7.742): ['the', 'big', 'sat', 'on', 'dog', 'a', 'cat', 'on']
Beam 3 (log-prob=-7.981): ['the', 'big', 'sat', 'on', 'dog', 'small', 'dog', '<EOS>']


## 3. Temperature Sampling

Scale the logits by a **temperature** $T$ before applying softmax:

$$P(x_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- $T = 1$: original distribution  
- $T \to 0$: approaches greedy (argmax) — distribution becomes peaky  
- $T > 1$: flattens the distribution — more uniform / random  

**Pros:** Simple knob to control diversity. Higher temperatures let the model explore multiple modes of the distribution, making it better at multi-modal generation.  
**Cons:** High $T$ introduces **incoherence** — low-probability (nonsensical) tokens get sampled too often. No principled way to set $T$; it affects all tokens equally regardless of their quality.

In [4]:
def temperature_sampling(prompt_ids: List[int], temperature: float = 1.0, max_len: int = 6) -> List[int]:
    """Sample from the full distribution after scaling logits by temperature."""
    generated = list(prompt_ids)
    eos_id = TOK2ID["<EOS>"]
    
    for _ in range(max_len):
        logits = dummy_logits(generated)
        # Scale logits by temperature
        scaled_logits = logits / temperature
        probs = softmax(scaled_logits)
        # Sample from the distribution
        next_id = int(np.random.choice(VOCAB_SIZE, p=probs))
        generated.append(next_id)
        if next_id == eos_id:
            break
    return generated

# Compare low vs high temperature
print("=== Temperature = 0.1 (nearly greedy) ===")
for run in range(3):
    ids = temperature_sampling(prompt, temperature=0.1)
    print(f"  {[ID2TOK[i] for i in ids]}")

print("\n=== Temperature = 1.0 (original) ===")
for run in range(3):
    ids = temperature_sampling(prompt, temperature=1.0)
    print(f"  {[ID2TOK[i] for i in ids]}")

print("\n=== Temperature = 2.0 (more random) ===")
for run in range(3):
    ids = temperature_sampling(prompt, temperature=2.0)
    print(f"  {[ID2TOK[i] for i in ids]}")

=== Temperature = 0.1 (nearly greedy) ===
  ['the', 'big', 'sat', '<EOS>']
  ['the', 'big', 'sat', '<EOS>']
  ['the', 'big', 'sat', '<EOS>']

=== Temperature = 1.0 (original) ===
  ['the', 'big', 'sat', '<EOS>']
  ['the', 'big', 'sat', 'on', 'small', '<EOS>']
  ['the', 'big', 'a', 'a', 'cat', 'big', 'mat', 'dog']

=== Temperature = 2.0 (more random) ===
  ['the', 'big', 'ran', 'big', 'ran', 'big', '<EOS>']
  ['the', 'big', 'sat', '<EOS>']
  ['the', 'big', 'dog', 'on', 'dog', 'on', 'on', 'dog']


## 4. Top-k Sampling

Only consider the **top-k** most probable tokens, zero out the rest, then re-normalize and sample:

1. Sort tokens by probability descending.
2. Keep only the top $k$ tokens.
3. Set all other probabilities to 0.
4. Re-normalize the remaining $k$ probabilities so they sum to 1.
5. Sample from this truncated distribution.

**Pros:** Prevents sampling from the long tail of low-quality tokens. Still stochastic, so it can capture multi-modal outputs.  
**Cons:** Fixed $k$ is a blunt instrument. If the model is **confident** (probability mass concentrated on 2 tokens), $k=50$ still allows 48 bad tokens. If the model is **uncertain** (probability spread across 100 good tokens), $k=50$ cuts off valid options. Doesn't adapt to the shape of the distribution.

In [5]:
def top_k_sampling(prompt_ids: List[int], k: int = 5, temperature: float = 1.0, max_len: int = 6) -> List[int]:
    """Sample from only the top-k most probable tokens."""
    generated = list(prompt_ids)
    eos_id = TOK2ID["<EOS>"]
    
    for _ in range(max_len):
        logits = dummy_logits(generated)
        scaled_logits = logits / temperature
        
        # Find the top-k indices
        top_k_indices = np.argsort(scaled_logits)[-k:]  # indices of top-k logits
        
        # Zero out everything outside top-k
        mask = np.full(VOCAB_SIZE, -np.inf)
        mask[top_k_indices] = scaled_logits[top_k_indices]
        
        probs = softmax(mask)
        next_id = int(np.random.choice(VOCAB_SIZE, p=probs))
        generated.append(next_id)
        if next_id == eos_id:
            break
    return generated

# Top-k with k=3
print("=== Top-k (k=3) ===")
for run in range(5):
    ids = top_k_sampling(prompt, k=3)
    print(f"  {[ID2TOK[i] for i in ids]}")

=== Top-k (k=3) ===
  ['the', 'big', 'sat', 'big', 'ran', 'cat', 'dog', 'sat']
  ['the', 'big', 'cat', 'mat', 'a', 'dog', 'cat', 'dog']
  ['the', 'big', 'cat', 'sat', 'small', 'on', 'on', 'cat']
  ['the', 'big', 'sat', 'big', 'on', 'on', 'the', 'the']
  ['the', 'big', 'dog', 'big', 'dog', 'mat', 'the', 'the']


## 5. Top-p (Nucleus) Sampling

Instead of a fixed $k$, keep the **smallest set of tokens** whose cumulative probability exceeds a threshold $p$:

1. Sort tokens by probability descending.
2. Compute the cumulative sum.
3. Keep all tokens up to and including the first one where cumulative probability $\geq p$.
4. Re-normalize and sample.

**Pros:** **Adapts** to the distribution shape. When the model is confident, the nucleus is small (1-2 tokens). When uncertain, it naturally expands to include many plausible tokens. This is the best strategy for multi-modal generation — it includes exactly the tokens that carry meaningful probability mass.  
**Cons:** Slightly more complex to implement. The threshold $p$ still requires tuning, but it's more robust than $k$.

In [6]:
def top_p_sampling(prompt_ids: List[int], p: float = 0.9, temperature: float = 1.0, max_len: int = 6) -> List[int]:
    """Nucleus sampling: keep smallest set of tokens with cumulative prob >= p."""
    generated = list(prompt_ids)
    eos_id = TOK2ID["<EOS>"]
    
    for _ in range(max_len):
        logits = dummy_logits(generated)
        scaled_logits = logits / temperature
        probs = softmax(scaled_logits)
        
        # Sort by probability descending
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        
        # Compute cumulative probabilities
        cumulative_probs = np.cumsum(sorted_probs)
        
        # Find cutoff: keep tokens up to where cumulative >= p
        cutoff_idx = np.searchsorted(cumulative_probs, p)
        # Include the token that crosses the threshold
        nucleus_indices = sorted_indices[:cutoff_idx + 1]
        
        # Zero out everything outside the nucleus
        mask = np.zeros(VOCAB_SIZE)
        mask[nucleus_indices] = probs[nucleus_indices]
        # Re-normalize
        mask /= mask.sum()
        
        next_id = int(np.random.choice(VOCAB_SIZE, p=mask))
        generated.append(next_id)
        if next_id == eos_id:
            break
    return generated

# Top-p with p=0.9
print("=== Top-p (p=0.9) ===")
for run in range(5):
    ids = top_p_sampling(prompt, p=0.9)
    print(f"  {[ID2TOK[i] for i in ids]}")

print("\n=== Top-p (p=0.5) — tighter nucleus ===")
for run in range(5):
    ids = top_p_sampling(prompt, p=0.5)
    print(f"  {[ID2TOK[i] for i in ids]}")

=== Top-p (p=0.9) ===
  ['the', 'big', 'on', 'cat', 'mat', 'cat', '<EOS>']
  ['the', 'big', 'dog', 'small', 'mat', 'ran', '<EOS>']
  ['the', 'big', 'a', 'the', 'cat', 'dog', 'ran', '<EOS>']
  ['the', 'big', 'dog', 'small', 'small', 'sat', 'the', 'ran']
  ['the', 'big', 'a', 'a', 'the', 'the', 'small', 'cat']

=== Top-p (p=0.5) — tighter nucleus ===
  ['the', 'big', 'cat', 'sat', '<EOS>']
  ['the', 'big', 'dog', 'big', 'cat', 'sat', '<EOS>']
  ['the', 'big', 'sat', 'big', 'ran', 'on', 'dog', 'a']
  ['the', 'big', 'sat', 'on', 'a', 'mat', '<EOS>']
  ['the', 'big', 'dog', 'big', 'dog', 'ran', 'dog', 'small']


## Summary: Multi-Modal Generation

| Strategy | Deterministic? | Multi-Modal? | Key Trade-off |
|---|---|---|---|
| **Greedy** | Yes | No — always picks the single most likely path | Fast but collapses diversity |
| **Beam Search** | Yes | Weak — top-B paths often converge | Better than greedy but still biased toward high-frequency outputs |
| **Temperature** | No | Yes, with high $T$ | Higher $T$ = more diversity but also more noise from the entire tail |
| **Top-k** | No | Yes | Cuts tail but fixed $k$ doesn't adapt to confidence |
| **Top-p (Nucleus)** | No | **Best** — adapts nucleus size to the distribution | Dynamically includes all "reasonable" tokens; best balance of quality and diversity |

### For Autonomous Driving / Waymo Context
When a model must predict **multiple plausible future trajectories** (e.g., a car might turn left OR go straight), the sampling strategy matters:
- Greedy/beam search would only predict the single most likely trajectory — dangerous if the car actually turns.
- **Top-p sampling** naturally captures the multi-modality: when the model is uncertain between two equally likely maneuvers, the nucleus includes both.
- Temperature can be combined with top-p to further control the spread.

### Common Practice
Most production LLMs use **top-p + temperature** together:
```
logits → scale by T → softmax → nucleus filter (top-p) → sample
```

In [7]:
def combined_sampling(prompt_ids: List[int], temperature: float = 0.8, top_k: int = None, 
                      top_p: float = 0.9, max_len: int = 6) -> List[int]:
    """Production-style sampling: temperature + optional top-k + top-p."""
    generated = list(prompt_ids)
    eos_id = TOK2ID["<EOS>"]
    
    for _ in range(max_len):
        logits = dummy_logits(generated)
        
        # Step 1: Temperature scaling
        scaled_logits = logits / temperature
        
        # Step 2: Optional top-k filtering
        if top_k is not None:
            top_k_indices = np.argsort(scaled_logits)[-top_k:]
            mask = np.full(VOCAB_SIZE, -np.inf)
            mask[top_k_indices] = scaled_logits[top_k_indices]
            scaled_logits = mask
        
        # Step 3: Convert to probabilities
        probs = softmax(scaled_logits)
        
        # Step 4: Top-p (nucleus) filtering
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        cumulative = np.cumsum(sorted_probs)
        cutoff = np.searchsorted(cumulative, top_p)
        nucleus = sorted_indices[:cutoff + 1]
        
        filtered = np.zeros(VOCAB_SIZE)
        filtered[nucleus] = probs[nucleus]
        filtered /= filtered.sum()
        
        next_id = int(np.random.choice(VOCAB_SIZE, p=filtered))
        generated.append(next_id)
        if next_id == eos_id:
            break
    return generated

# Combined: temperature=0.8, top-p=0.9
print("=== Combined (T=0.8, top-p=0.9) ===")
for run in range(5):
    ids = combined_sampling(prompt, temperature=0.8, top_p=0.9)
    print(f"  {[ID2TOK[i] for i in ids]}")

=== Combined (T=0.8, top-p=0.9) ===
  ['the', 'big', 'dog', 'small', 'small', 'cat', 'a', 'on']
  ['the', 'big', 'sat', 'big', 'cat', 'sat', 'the', 'on']
  ['the', 'big', 'a', 'a', 'mat', 'cat', 'cat', 'cat']
  ['the', 'big', 'sat', 'mat', 'small', 'big', 'small', 'mat']
  ['the', 'big', 'cat', 'sat', 'the', 'on', 'the', 'sat']
